# Module 3 · Wrapping FastAPI in Docker
### FastAPI for ML Engineers — Bootcamp Session (Part 3 of 4)

**Time budget:** ~60 minutes
**Prereq:** Modules 1–2. **You'll need Docker Desktop / Docker Engine installed locally** to run the `!docker ...` cells in this notebook — they won't run inside a hosted/sandboxed notebook environment with no Docker daemon (e.g. plain Colab). Everything else (writing files, understanding the Dockerfile) works anywhere.

## Agenda
| Time | Topic |
|---|---|
| 0:00–0:10 | Why Docker for ML serving |
| 0:10–0:15 | Project layout & `requirements.txt` |
| 0:15–0:30 | Anatomy of a Dockerfile, layer caching |
| 0:30–0:40 | Build & run the container, call it like a real client |
| 0:40–0:50 | Env vars/secrets, `HEALTHCHECK`, logging, image size |
| 0:50–1:00 | Exercise (multi-stage build) + wrap-up |

## Learning objectives
- Explain why we containerize model-serving APIs at all
- Write a correct, reasonably efficient Dockerfile for a FastAPI + scikit-learn service
- Build, run, and call the container locally
- Handle environment variables/secrets correctly (never bake them into the image)
- Know what Vertex AI (Module 4) will expect from this exact container

## 3.1 Why Docker for ML serving?

Without a container, "it works on my machine" is a real, constant problem for ML services specifically, because:
- Your model was trained with a **specific** version of scikit-learn/numpy/torch — a mismatch on the serving side can silently change predictions or crash on load.
- System-level dependencies (BLAS libraries, CUDA versions, `libgomp`) differ across machines.
- You need the **exact same artifact** to move from your laptop → staging → production → a managed platform like Vertex AI, without "reinstalling and hoping."

Docker solves this by packaging **the OS layer, the Python runtime, your dependencies, your code, and (often) your model artifact** into one immutable image. That image is what Vertex AI Custom Container serving actually runs in Module 4 — so everything you build here is not a detour, it's the direct prerequisite.

## 3.2 Project layout

We'll create a small, real project directory from inside this notebook using the `%%writefile` magic, so by the end you have an actual folder you could `cd` into and `docker build`.

```
fastapi_docker_project/
├── app/
│   ├── main.py              <- FastAPI app (same shape as Module 2)
│   └── model_artifacts/
│       └── model.joblib     <- trained model
├── requirements.txt
└── Dockerfile
```

In [ ]:
import os

PROJECT_DIR = "/home/claude/fastapi_docker_project"
os.makedirs(f"{PROJECT_DIR}/app/model_artifacts", exist_ok=True)
print("project dir ready at", PROJECT_DIR)

### Re-create (or reuse) the tiny model from Module 2, so this notebook is self-contained

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=2000, n_features=3, n_informative=3,
                            n_redundant=0, random_state=42)
clf = LogisticRegression().fit(X, y)

MODEL_PATH = f"{PROJECT_DIR}/app/model_artifacts/model.joblib"
joblib.dump(clf, MODEL_PATH)
print("model saved to", MODEL_PATH)

In [ ]:
%%writefile /home/claude/fastapi_docker_project/requirements.txt
fastapi==0.115.0
uvicorn[standard]==0.30.6
scikit-learn==1.5.2
joblib==1.4.2
numpy==1.26.4
pydantic==2.9.2

> **Pin your versions.** `fastapi` with no version pin means the image you build today can behave differently in six months when a new release ships. This is *especially* important for the ML libraries — an unpinned `scikit-learn` can silently fail to load a model pickled with an older version.

In [ ]:
%%writefile /home/claude/fastapi_docker_project/app/main.py
import os
import joblib
import numpy as np
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

MODEL_PATH = os.path.join(os.path.dirname(__file__), "model_artifacts", "model.joblib")
MODEL_VERSION = "fraud_v1_logreg"

ML_MODELS = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    ML_MODELS["fraud_model"] = joblib.load(MODEL_PATH)
    yield
    ML_MODELS.clear()

app = FastAPI(title="Fraud Model Serving API", version="1.0.0", lifespan=lifespan)


class PredictionRequest(BaseModel):
    feature_1: float = Field(..., description="Account age in years")
    feature_2: float = Field(..., description="Number of transactions in last 24h")
    feature_3: float = Field(..., description="Average transaction amount")


class PredictionResponse(BaseModel):
    prediction: int
    probability: float
    model_version: str


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": "fraud_model" in ML_MODELS}


@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    model = ML_MODELS.get("fraud_model")
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")

    features = np.array([[request.feature_1, request.feature_2, request.feature_3]])
    try:
        pred = int(model.predict(features)[0])
        proba = float(model.predict_proba(features)[0][pred])
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Inference failed: {type(e).__name__}")

    return PredictionResponse(prediction=pred, probability=proba, model_version=MODEL_VERSION)

## 3.3 The Dockerfile, line by line

Key decisions, called out deliberately (these are the exact things a reviewer will ask about in a PR):

- **`python:3.11-slim`** as the base — full `python:3.11` includes a lot of build tooling and OS packages you don't need at runtime; `slim` is a good default balance of size vs. compatibility.
- **Copy `requirements.txt` *before* the rest of the code**, then `RUN pip install`. Docker caches layers — if only your application code changes (not dependencies), this ordering means the (slow) `pip install` layer is reused from cache instead of re-running every build.
- **Run as a non-root user.** Containers running as root are a real security surface; drop privileges after installing dependencies.
- **`EXPOSE 8080`** — not 8000. Vertex AI's custom container serving expects your container to listen on `$PORT` (which it sets, defaulting to 8080) — more on this in Module 4. Building the habit now saves you a debugging session later.
- **`CMD` uses `uvicorn` directly**, binding to `0.0.0.0` (not `127.0.0.1` — inside a container, `localhost` only accepts traffic from inside the container itself, which means the host can never reach it) and reading the port from an environment variable with a sane default.

In [ ]:
%%writefile /home/claude/fastapi_docker_project/Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Install dependencies first for better layer caching
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Now copy application code + model artifact
COPY app/ ./app/

# Run as a non-root user
RUN useradd --create-home appuser
USER appuser

# Vertex AI (and most managed platforms) inject $PORT; default to 8080 for local runs
ENV PORT=8080
EXPOSE 8080

HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 CMD python -c "import urllib.request, os; urllib.request.urlopen('http://localhost:' + os.environ.get('PORT', '8080') + '/health')" || exit 1

CMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT}"]

## 3.4 Build the image

Run this from a terminal (or the `!docker` cell below) with Docker running locally:

In [ ]:
%%bash
cd /home/claude/fastapi_docker_project
docker build -t fraud-model-api:latest .

## 3.5 Run the container and call it

`-p 8000:8080` maps **host** port 8000 to the **container's** port 8080 (the one we set `$PORT` to and `EXPOSE`d). This mapping is exactly why it doesn't matter that the container listens on 8080 internally — you can expose it as whatever port makes sense on your host.

In [ ]:
%%bash
docker run -d --name fraud-api -p 8000:8080 fraud-model-api:latest
sleep 2
docker ps --filter "name=fraud-api"

In [ ]:
import requests

resp = requests.get("http://localhost:8000/health")
print(resp.status_code, resp.json())

resp = requests.post(
    "http://localhost:8000/predict",
    json={"feature_1": 2.0, "feature_2": 5, "feature_3": 10.0},
)
print(resp.status_code, resp.json())

In [ ]:
%%bash
# Tear down when you're done experimenting
docker stop fraud-api
docker rm fraud-api

## 3.6 Environment variables & secrets — don't bake them into the image

Anything in your `Dockerfile` (via `ENV` or `ARG`) or `COPY`'d into the image is **visible to anyone who can pull or inspect that image** — `docker history` and `docker inspect` will show `ENV` values, and copied files are just sitting in a layer. Never do this:

```dockerfile
# DON'T DO THIS
ENV DATABASE_PASSWORD=hunter2
```

Instead, inject secrets **at runtime**, not build time:

```bash
docker run -p 8000:8080 --env DATABASE_PASSWORD=hunter2 fraud-model-api:latest
# or, better, from a local file that's gitignored:
docker run -p 8000:8080 --env-file .env.local fraud-model-api:latest
```

In production on GCP, you'd typically pull the secret from **Secret Manager** at container startup rather than passing it as a plain env var at all — but env-var injection at `docker run` / at the orchestrator level (Vertex AI lets you set env vars on the deployed model) is the baseline pattern to know.

## 3.7 Logging

Don't write logs to a file inside the container — it disappears when the container is removed, and nothing outside can see it live. Log to **stdout/stderr**; Docker (and every cloud platform built on it, including Vertex AI and Cloud Logging) captures that automatically.

```python
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("fraud_api")

@app.post("/predict")
def predict(request: PredictionRequest):
    logger.info(f"prediction request received: {request.model_dump()}")
    ...
```

`docker logs fraud-api` (or `docker logs -f fraud-api` to follow) shows you exactly this output while developing locally.

## 3.8 Multi-stage builds (reducing image size)

For classical ML (scikit-learn, XGBoost) the simple Dockerfile above is usually good enough. For heavier stacks (PyTorch, compiling native extensions), a **multi-stage build** compiles/installs in one stage and copies only the final artifacts into a slim final image — smaller images mean faster deploys and a smaller attack surface.

```dockerfile
# ---- builder stage ----
FROM python:3.11 AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/install -r requirements.txt

# ---- final stage ----
FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /install /usr/local/lib/python3.11/site-packages
COPY app/ ./app/
ENV PORT=8080
EXPOSE 8080
CMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT}"]
```

The `builder` stage's OS build tools, caches, and intermediate files never make it into the final image — only the installed packages do.

## Exercise 3 (10 min)

1. Rewrite this notebook's `Dockerfile` as a multi-stage build using the pattern above, build it, and compare `docker images` sizes between the single-stage and multi-stage versions.
2. Add a second endpoint's worth of dependency (e.g. `pandas`) to `requirements.txt`, rebuild, and confirm via `docker history fraud-model-api:latest` which layer grew.
3. Bonus: convert the plain `print`/no logging in `main.py` to use the `logging` module, rebuild, run, hit `/predict` a few times, and confirm you can see structured log lines with `docker logs fraud-api`.

In [ ]:
%%bash
# Scaffold for exercise 3.1 — compare image sizes once you've built both versions
docker images fraud-model-api

## Recap

- Docker gives you a reproducible, portable unit that behaves identically from your laptop to a managed platform.
- Layer ordering (deps before code) speeds up rebuilds; pin your dependency versions.
- Bind to `0.0.0.0`, listen on `$PORT` (default 8080) — this is exactly what Vertex AI custom containers expect.
- Never bake secrets into the image; inject them at runtime.
- Log to stdout/stderr, not files.

**Next up — Module 4:** pushing this image to Artifact Registry and deploying it as a live prediction endpoint on Vertex AI.